# 🏭 Üretim Anomali Tespiti
## Sliding Window + Isolation Forest
> **Veri:** `excel_partiler_total.xlsx` &nbsp;|&nbsp; **7 Sensör** &nbsp;|&nbsp; **12 Parti (6N + 6A)**

### Adımlar
1. Kurulum & İçe Aktarma
2. Veri Yükleme
3–5. EDA (Genel Bakış · Zaman Serileri · KDE Dağılımlar)
6. Sliding Window Feature Extraction
7. Eğitim / Test Ayrımı
8. Model Eğitimi (Isolation Forest)
9. Eşik Optimizasyonu
10. Performans Metrikleri
11. Anomali Skor Zaman Serileri
12. Sensör Katkı Analizi
13. Özet Tablo & Dosya İndirme

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   ÜRETİM ANOMALİ TESPİTİ — Google Colab Notebook                       ║
# ║   Sliding Window + Isolation Forest                                      ║
# ║   Veri: excel_partiler_total.xlsx                                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝


### HÜCRE 1: Kurulum & İçe Aktarma

In [ ]:
# ── HÜCRE 1: Kurulum & İçe Aktarma ─────────────────────────────────────────
# Colab'da çalıştır — yerel ortamda zaten kuruluysa geç

!pip install -q openpyxl scikit-learn matplotlib seaborn pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    precision_recall_curve, roc_curve, average_precision_score
)
from sklearn.inspection import permutation_importance
import os
from google.colab import files   # Colab otomatik indirme için

# Stil ayarları
plt.rcParams.update({
    "figure.facecolor": "#0F1117",
    "axes.facecolor":   "#1A1D27",
    "axes.edgecolor":   "#3A3F54",
    "axes.labelcolor":  "#C8CDD8",
    "xtick.color":      "#8891A5",
    "ytick.color":      "#8891A5",
    "text.color":       "#E2E5EE",
    "grid.color":       "#2A2F42",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
    "font.family":      "monospace",
    "axes.titlesize":   11,
    "axes.titleweight": "bold",
    "axes.titlecolor":  "#FFFFFF",
    "figure.dpi":       130,
})

PALETTE_NORMAL = "#3DD68C"   # yeşil — normal
PALETTE_ANOM   = "#FF5370"   # kırmızı — anormal
PALETTE_SCORE  = "#7C83FD"   # mor — skor çizgisi
PALETTE_THRESH = "#FFB547"   # turuncu — eşik

print("✓ Kurulum ve içe aktarma tamamlandı.")


### HÜCRE 2: Veri Yükleme

In [ ]:
# ── HÜCRE 2: Veri Yükleme ───────────────────────────────────────────────────

# Colab'a dosya yükle (aşağıdaki hücreyi çalıştır, dosya seçiciyi kullan)
uploaded = files.upload()   # excel_partiler_total.xlsx seç
FILENAME = list(uploaded.keys())[0]

df_raw = pd.read_excel(FILENAME, sheet_name=0)

# ── VERİ TEMİZLEME ──────────────────────────────────────────────────────────
# Sensör sütunlarını sayısala zorla; kirli değerleri (örn. '"') NaN yap,
# NaN satırları önceki geçerli değerle doldur (ffill), kalan NaN → 0
_SENSOR_RAW = [
    "S1_AK_Sicakligi", "S2_BK1_Sicakligi", "S3_BK2_Sicaklik",
    "S4_AK_Seviyesi",  "S5_Iletkenlik",     "S6_BK1_Seviyesi",
    "S7_BK2_Seviyesi"
]
for _col in _SENSOR_RAW:
    if _col in df_raw.columns:
        df_raw[_col] = pd.to_numeric(df_raw[_col], errors="coerce")

df_raw[_SENSOR_RAW] = (
    df_raw[_SENSOR_RAW]
    .ffill()   # önceki geçerli değerle doldur
    .fillna(0) # hâlâ NaN kalırsa 0
)
_dirty = df_raw[_SENSOR_RAW].isna().sum().sum()
print(f"✓ Veri temizlendi — kalan NaN: {_dirty}")

# Sensör ve meta sütunlar
SENSOR_COLS = [
    "S1_AK_Sicakligi", "S2_BK1_Sicakligi", "S3_BK2_Sicaklik",
    "S4_AK_Seviyesi",  "S5_Iletkenlik",     "S6_BK1_Seviyesi",
    "S7_BK2_Seviyesi"
]
SENSOR_LABELS = [
    "AK Sıcaklığı", "BK1 Sıcaklığı", "BK2 Sıcaklık",
    "AK Seviyesi",  "İletkenlik",     "BK1 Seviyesi",
    "BK2 Seviyesi"
]

# Etiket: Kusur sütunu (0=normal, 1=anormal)
df_raw["label"] = df_raw["Kusur"].astype(int)

# Parti bazında etiket tablosu
batch_labels = (
    df_raw.groupby("batch_id")["label"]
    .first()
    .reset_index()
    .rename(columns={"label": "label"})
)
batch_labels["durum"] = batch_labels["label"].map({0: "Normal", 1: "Anormal"})

print(f"✓ Veri yüklendi: {len(df_raw):,} satır | {df_raw['batch_id'].nunique()} parti")
print(f"  Normal  : {(batch_labels['label']==0).sum()} parti")
print(f"  Anormal : {(batch_labels['label']==1).sum()} parti")
print()
print(batch_labels.to_string(index=False))


### HÜCRE 3: EDA — Genel Bakış

In [ ]:
# ── HÜCRE 3: EDA — Genel Bakış ──────────────────────────────────────────────

fig = plt.figure(figsize=(18, 10))
fig.suptitle("EDA — Parti Genel Bakış", fontsize=14, color="#FFFFFF", y=1.01)

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.3)

# 3a: Parti uzunlukları
ax1 = fig.add_subplot(gs[0, 0])
lengths = df_raw.groupby("batch_id")["t"].max().reset_index()
lengths = lengths.merge(batch_labels, on="batch_id")
colors_bar = [PALETTE_ANOM if l == 1 else PALETTE_NORMAL
              for l in lengths["label"]]
bars = ax1.barh(lengths["batch_id"].astype(str), lengths["t"],
                color=colors_bar, alpha=0.85, edgecolor="none")
ax1.set_xlabel("Ölçüm Sayısı (dakika)")
ax1.set_title("Parti Uzunlukları")
ax1.axvline(lengths["t"].mean(), color=PALETTE_THRESH, ls="--", lw=1.2,
            label=f"Ort. {lengths['t'].mean():.0f}")
ax1.legend(fontsize=7)
# Değer etiketi
for bar, val in zip(bars, lengths["t"]):
    ax1.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             str(val), va="center", fontsize=7, color="#C8CDD8")
ax1.set_xlim(0, lengths["t"].max() * 1.15)

# 3b: Sensör bazında boxplot (normal vs anormal)
ax2 = fig.add_subplot(gs[0, 1])
melt = df_raw[SENSOR_COLS + ["label"]].copy()
melt.columns = SENSOR_LABELS + ["label"]
melt_long = melt.melt(id_vars="label", var_name="Sensör", value_name="Değer")
melt_long["Durum"] = melt_long["label"].map({0: "Normal", 1: "Anormal"})
sns.boxplot(
    data=melt_long, x="Sensör", y="Değer", hue="Durum",
    palette={"Normal": PALETTE_NORMAL, "Anormal": PALETTE_ANOM},
    ax=ax2, linewidth=0.7, fliersize=1.5, width=0.6,
    boxprops=dict(alpha=0.85)
)
ax2.set_title("Sensör Dağılımı: Normal vs Anormal")
ax2.set_xlabel("")
ax2.tick_params(axis="x", rotation=25)
ax2.legend(fontsize=7)

# 3c: Korelasyon matrisi (normal partiler)
ax3 = fig.add_subplot(gs[1, 0])
corr = df_raw[df_raw["label"] == 0][SENSOR_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, ax=ax3, mask=mask, cmap=cmap, center=0,
            annot=True, fmt=".2f", annot_kws={"size": 7},
            linewidths=0.4, square=True,
            xticklabels=SENSOR_LABELS, yticklabels=SENSOR_LABELS,
            cbar_kws={"shrink": 0.7})
ax3.set_title("Sensör Korelasyonu (Normal Partiler)")
ax3.tick_params(axis="x", rotation=30, labelsize=7)
ax3.tick_params(axis="y", rotation=0, labelsize=7)

# 3d: Sensör ortalama profili (normal vs anormal)
ax4 = fig.add_subplot(gs[1, 1])
normal_mean = df_raw[df_raw["label"] == 0][SENSOR_COLS].mean()
anom_mean   = df_raw[df_raw["label"] == 1][SENSOR_COLS].mean()
x = np.arange(len(SENSOR_LABELS))
w = 0.35
ax4.bar(x - w/2, normal_mean, w, label="Normal",
        color=PALETTE_NORMAL, alpha=0.85, edgecolor="none")
ax4.bar(x + w/2, anom_mean,   w, label="Anormal",
        color=PALETTE_ANOM,   alpha=0.85, edgecolor="none")
ax4.set_xticks(x)
ax4.set_xticklabels(SENSOR_LABELS, rotation=25, fontsize=8)
ax4.set_title("Sensör Ortalamaları: Normal vs Anormal")
ax4.legend(fontsize=8)
ax4.set_ylabel("Ortalama Değer")

plt.savefig("eda_genel_bakis.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("eda_genel_bakis.png")
print("✓ EDA Genel Bakış indirildi.")


### HÜCRE 4: EDA — Sensör Zaman Serileri

In [ ]:
# ── HÜCRE 4: EDA — Sensör Zaman Serileri ────────────────────────────────────

fig, axes = plt.subplots(4, 2, figsize=(18, 16))
fig.suptitle("EDA — Sensör Zaman Serileri (Tüm Partiler)", fontsize=13,
             color="#FFFFFF")
axes = axes.flatten()

for i, (sensor, label_tr) in enumerate(zip(SENSOR_COLS, SENSOR_LABELS)):
    ax = axes[i]
    for _, row in batch_labels.iterrows():
        bid   = row["batch_id"]
        lbl   = row["label"]
        bdata = df_raw[df_raw["batch_id"] == bid].sort_values("t")
        color = PALETTE_ANOM if lbl == 1 else PALETTE_NORMAL
        alpha = 0.55 if lbl == 1 else 0.70
        lw    = 1.0  if lbl == 1 else 0.8
        ax.plot(bdata["t"], bdata[sensor], color=color,
                alpha=alpha, lw=lw, label=str(bid) if i == 0 else "")
    ax.set_title(label_tr)
    ax.set_xlabel("t (ölçüm sırası)", fontsize=7)
    ax.set_ylabel("Değer", fontsize=7)
    ax.tick_params(labelsize=7)
    # Legend sadece ilk panel için
    if i == 0:
        from matplotlib.lines import Line2D
        legend_els = [
            Line2D([0], [0], color=PALETTE_NORMAL, lw=2, label="Normal"),
            Line2D([0], [0], color=PALETTE_ANOM,   lw=2, label="Anormal"),
        ]
        ax.legend(handles=legend_els, fontsize=8, loc="upper right")

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig("eda_zaman_serileri.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("eda_zaman_serileri.png")
print("✓ Zaman serileri grafiği indirildi.")


### HÜCRE 5: EDA — İstatistiksel Özet & Dağılımlar

In [ ]:
# ── HÜCRE 5: EDA — İstatistiksel Özet & Dağılımlar ──────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("EDA — Sensör Dağılımları (KDE)", fontsize=13, color="#FFFFFF")
axes = axes.flatten()

normal_data = df_raw[df_raw["label"] == 0]
anom_data   = df_raw[df_raw["label"] == 1]

for i, (sensor, label_tr) in enumerate(zip(SENSOR_COLS, SENSOR_LABELS)):
    ax = axes[i]
    sns.kdeplot(normal_data[sensor].dropna(), ax=ax,
                color=PALETTE_NORMAL, fill=True, alpha=0.35, lw=1.5,
                label="Normal")
    sns.kdeplot(anom_data[sensor].dropna(), ax=ax,
                color=PALETTE_ANOM, fill=True, alpha=0.35, lw=1.5,
                label="Anormal")
    ax.set_title(label_tr, fontsize=9)
    ax.set_xlabel("Değer", fontsize=7)
    ax.set_ylabel("Yoğunluk", fontsize=7)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig("eda_kde_dagilimlar.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("eda_kde_dagilimlar.png")
print("✓ KDE dağılım grafiği indirildi.")

# Sayısal özet tablosu
print("\n── SENSÖR İSTATİSTİKLERİ (Normal) ──")
print(normal_data[SENSOR_COLS].describe().round(2).to_string())
print("\n── SENSÖR İSTATİSTİKLERİ (Anormal) ──")
print(anom_data[SENSOR_COLS].describe().round(2).to_string())


### HÜCRE 6: Sliding Window Feature Extraction

In [ ]:
# ── HÜCRE 6: Sliding Window Feature Extraction ──────────────────────────────

WINDOW_SIZE = 30   # dk — başlangıç değeri, optimize edilecek

def extract_window_features(df, window_size=WINDOW_SIZE):
    """
    Her zaman noktası için son window_size adımlık pencereyi flatten eder.
    Çıktı: feature matrix + meta dataframe
    """
    results = []
    for batch_id, group in df.groupby("batch_id"):
        group   = group.sort_values("t").reset_index(drop=True)
        sensors = group[SENSOR_COLS].values      # (n, 7)
        label   = group["label"].iloc[0]
        n_rows  = len(sensors)

        for t_idx in range(window_size - 1, n_rows):
            window = sensors[t_idx - window_size + 1 : t_idx + 1]  # (30, 7)
            flat   = window.flatten()                               # (210,)
            results.append({
                "batch_id": batch_id,
                "t":        group["t"].iloc[t_idx],
                "label":    label,
                "features": flat,
            })

    feat_df = pd.DataFrame(results)
    X       = np.vstack(feat_df["features"].values)
    meta    = feat_df[["batch_id", "t", "label"]].reset_index(drop=True)

    n_feat = window_size * len(SENSOR_COLS)
    feat_names = [
        f"w{i // len(SENSOR_COLS)}_S{(i % len(SENSOR_COLS)) + 1}"
        for i in range(n_feat)
    ]
    print(f"✓ Feature extraction tamamlandı")
    print(f"  Window size   : {window_size} dk")
    print(f"  Feature sayısı: {n_feat} ({window_size} × {len(SENSOR_COLS)})")
    print(f"  Toplam örnek  : {len(X):,}")
    return X, meta, feat_names


X_all, meta_all, feat_names = extract_window_features(df_raw)


### HÜCRE 7: Eğitim / Test Ayrımı

In [ ]:
# ── HÜCRE 7: Eğitim / Test Ayrımı ───────────────────────────────────────────

normal_batch_ids = batch_labels[batch_labels["label"] == 0]["batch_id"].tolist()
anom_batch_ids   = batch_labels[batch_labels["label"] == 1]["batch_id"].tolist()

# Tüm normal partiler → eğitim
# Tüm anormal partiler → test (normal partiler de test'e dahil → gerçekçi değerlendirme)
train_mask = meta_all["batch_id"].isin(normal_batch_ids)
test_mask  = pd.Series([True] * len(meta_all))   # tümü test'te görülsün

X_train = X_all[train_mask]
X_test  = X_all                                   # 12 parti
meta_test = meta_all.copy()

print(f"✓ Eğitim seti: {len(X_train):,} örnek (6 normal parti)")
print(f"✓ Test seti  : {len(X_test):,} örnek (12 parti — 6N + 6A)")


### HÜCRE 8: Model Eğitimi

In [ ]:
# ── HÜCRE 8: Model Eğitimi ───────────────────────────────────────────────────

scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model = IsolationForest(
    n_estimators  = 300,
    contamination = 0.05,    # eğitim setinde beklenen anomali oranı
    max_samples   = "auto",
    random_state  = 42,
    n_jobs        = -1,
)
model.fit(X_train_scaled)

# Skor hesaplama (yüksek → daha anomalik)
raw_scores  = model.score_samples(X_test_scaled)
anom_scores = -raw_scores
if_preds    = model.predict(X_test_scaled)        # -1=anomali, 1=normal
binary_pred = np.where(if_preds == -1, 1, 0)

meta_test = meta_test.copy()
meta_test["anomaly_score"] = anom_scores
meta_test["pred_binary"]   = binary_pred

print(f"✓ Model eğitimi tamamlandı (300 ağaç)")
print(f"  Toplam anomali tahmini: {binary_pred.sum():,} örnek "
      f"({binary_pred.sum()/len(binary_pred)*100:.1f}%)")


### HÜCRE 9: Eşik Optimizasyonu

In [ ]:
# ── HÜCRE 9: Eşik Optimizasyonu ─────────────────────────────────────────────
# Parti düzeyi etiketlerle en iyi eşiği bul

def batch_level_eval(meta_df, threshold):
    """Verilen eşikte parti bazında tahmin ve gerçek etiketi karşılaştır."""
    grp = meta_df.groupby("batch_id").agg(
        true_label    = ("label", "first"),
        max_score     = ("anomaly_score", "max"),
        mean_score    = ("anomaly_score", "mean"),
    ).reset_index()
    grp["pred_label"] = (grp["max_score"] > threshold).astype(int)
    return grp

# Eşik tarama
thresholds = np.percentile(anom_scores, np.arange(70, 99, 0.5))
results = []
for thr in thresholds:
    grp = batch_level_eval(meta_test, thr)
    try:
        auc = roc_auc_score(grp["true_label"], grp["max_score"])
    except Exception:
        auc = 0.0
    # F1 benzeri: TP / (TP + 0.5*(FP+FN))
    tp = ((grp["pred_label"] == 1) & (grp["true_label"] == 1)).sum()
    fp = ((grp["pred_label"] == 1) & (grp["true_label"] == 0)).sum()
    fn = ((grp["pred_label"] == 0) & (grp["true_label"] == 1)).sum()
    f1 = tp / (tp + 0.5 * (fp + fn) + 1e-9)
    results.append({"threshold": thr, "f1": f1, "auc": auc, "tp": tp, "fp": fp, "fn": fn})

thr_df = pd.DataFrame(results)
best_row = thr_df.loc[thr_df["f1"].idxmax()]
BEST_THRESHOLD = best_row["threshold"]

print(f"✓ Eşik optimizasyonu tamamlandı")
print(f"  En iyi eşik : {BEST_THRESHOLD:.4f}")
print(f"  F1 skoru    : {best_row['f1']:.3f}")
print(f"  ROC-AUC     : {best_row['auc']:.3f}")
print(f"  TP: {int(best_row['tp'])}  FP: {int(best_row['fp'])}  FN: {int(best_row['fn'])}")


### HÜCRE 10: Performans Metrikleri & Görselleştirme

In [ ]:
# ── HÜCRE 10: Performans Metrikleri & Görselleştirme ────────────────────────

batch_results = batch_level_eval(meta_test, BEST_THRESHOLD)

fig = plt.figure(figsize=(18, 12))
fig.suptitle("Performans Metrikleri — Isolation Forest", fontsize=14,
             color="#FFFFFF", y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 10a: Confusion Matrix
ax1 = fig.add_subplot(gs[0, 0])
cm = confusion_matrix(batch_results["true_label"], batch_results["pred_label"])
sns.heatmap(cm, annot=True, fmt="d", ax=ax1, cmap="YlOrRd",
            xticklabels=["Normal", "Anormal"],
            yticklabels=["Normal", "Anormal"],
            annot_kws={"size": 16, "weight": "bold"},
            linewidths=0.5, cbar=False)
ax1.set_title("Confusion Matrix (Parti Düzeyi)")
ax1.set_xlabel("Tahmin", fontsize=9)
ax1.set_ylabel("Gerçek", fontsize=9)

# 10b: ROC Curve
ax2 = fig.add_subplot(gs[0, 1])
fpr, tpr, _ = roc_curve(batch_results["true_label"],
                          batch_results["max_score"])
auc_val = roc_auc_score(batch_results["true_label"],
                         batch_results["max_score"])
ax2.plot(fpr, tpr, color=PALETTE_SCORE, lw=2,
         label=f"ROC AUC = {auc_val:.3f}")
ax2.plot([0, 1], [0, 1], "w--", lw=0.8, alpha=0.5)
ax2.fill_between(fpr, tpr, alpha=0.15, color=PALETTE_SCORE)
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Eğrisi (Parti Düzeyi)")
ax2.legend(fontsize=9)
ax2.set_xlim([-0.02, 1.02])
ax2.set_ylim([-0.02, 1.05])

# 10c: Precision-Recall
ax3 = fig.add_subplot(gs[0, 2])
prec, rec, _ = precision_recall_curve(batch_results["true_label"],
                                       batch_results["max_score"])
ap = average_precision_score(batch_results["true_label"],
                               batch_results["max_score"])
ax3.plot(rec, prec, color=PALETTE_ANOM, lw=2, label=f"AP = {ap:.3f}")
ax3.fill_between(rec, prec, alpha=0.15, color=PALETTE_ANOM)
ax3.set_xlabel("Recall")
ax3.set_ylabel("Precision")
ax3.set_title("Precision-Recall Eğrisi")
ax3.legend(fontsize=9)
ax3.set_xlim([-0.02, 1.02])
ax3.set_ylim([-0.02, 1.05])

# 10d: Eşik vs F1
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(thr_df["threshold"], thr_df["f1"],
         color=PALETTE_SCORE, lw=1.8)
ax4.axvline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--", lw=1.5,
            label=f"Best = {BEST_THRESHOLD:.3f}")
ax4.set_xlabel("Eşik Değeri")
ax4.set_ylabel("F1 Skoru")
ax4.set_title("Eşik Optimizasyonu")
ax4.legend(fontsize=8)

# 10e: Parti bazında max anomali skoru
ax5 = fig.add_subplot(gs[1, 1])
br = batch_results.sort_values("max_score", ascending=True)
colors_br = [PALETTE_ANOM if l == 1 else PALETTE_NORMAL
             for l in br["true_label"]]
bars = ax5.barh(br["batch_id"].astype(str), br["max_score"],
                color=colors_br, alpha=0.85, edgecolor="none")
ax5.axvline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--", lw=1.5,
            label=f"Eşik = {BEST_THRESHOLD:.3f}")
ax5.set_xlabel("Max Anomali Skoru")
ax5.set_title("Parti Bazında Max Skor")
ax5.legend(fontsize=8)
from matplotlib.patches import Patch
legend_els = [
    Patch(color=PALETTE_NORMAL, label="Normal"),
    Patch(color=PALETTE_ANOM,   label="Anormal"),
]
ax5.legend(handles=legend_els + [
    plt.Line2D([0],[0], color=PALETTE_THRESH, ls="--", lw=1.5,
               label=f"Eşik={BEST_THRESHOLD:.3f}")
], fontsize=7)

# 10f: Skor dağılımı (Normal vs Anormal)
ax6 = fig.add_subplot(gs[1, 2])
score_normal = meta_test[meta_test["label"] == 0]["anomaly_score"]
score_anom   = meta_test[meta_test["label"] == 1]["anomaly_score"]
ax6.hist(score_normal, bins=60, color=PALETTE_NORMAL, alpha=0.6,
         density=True, label="Normal", edgecolor="none")
ax6.hist(score_anom, bins=60, color=PALETTE_ANOM, alpha=0.6,
         density=True, label="Anormal", edgecolor="none")
ax6.axvline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--", lw=1.5,
            label=f"Eşik")
ax6.set_xlabel("Anomali Skoru")
ax6.set_ylabel("Yoğunluk")
ax6.set_title("Skor Dağılımı: Normal vs Anormal")
ax6.legend(fontsize=8)

plt.savefig("performans_metrikleri.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("performans_metrikleri.png")
print("✓ Performans metrikleri grafiği indirildi.")

# Yazılı rapor
print("\n" + "═" * 55)
print("  PERFORMANS RAPORU — PARTİ DÜZEYİ")
print("═" * 55)
print(classification_report(
    batch_results["true_label"],
    batch_results["pred_label"],
    target_names=["Normal", "Anormal"],
    zero_division=0,
))
print(f"ROC-AUC        : {auc_val:.4f}")
print(f"Avg Precision  : {ap:.4f}")
print(f"Kullanılan Eşik: {BEST_THRESHOLD:.4f}")


### HÜCRE 11: Anomali Skor Zaman Serileri

In [ ]:
# ── HÜCRE 11: Anomali Skor Zaman Serileri ───────────────────────────────────

batches_sorted = (
    batch_results.sort_values(["true_label", "batch_id"])["batch_id"].tolist()
)
n_batches = len(batches_sorted)
n_cols    = 3
n_rows    = int(np.ceil(n_batches / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
fig.suptitle("Anomali Skoru Zaman Serileri — Tüm Partiler",
             fontsize=14, color="#FFFFFF")
axes = axes.flatten()

for i, bid in enumerate(batches_sorted):
    ax   = axes[i]
    bdat = meta_test[meta_test["batch_id"] == bid].sort_values("t")
    lbl  = batch_results.loc[batch_results["batch_id"] == bid, "true_label"].iloc[0]
    pred = batch_results.loc[batch_results["batch_id"] == bid, "pred_label"].iloc[0]
    color = PALETTE_ANOM if lbl == 1 else PALETTE_NORMAL

    ax.fill_between(bdat["t"], bdat["anomaly_score"],
                    alpha=0.18, color=color)
    ax.plot(bdat["t"], bdat["anomaly_score"],
            color=PALETTE_SCORE, lw=1.2, alpha=0.9)
    ax.axhline(BEST_THRESHOLD, color=PALETTE_THRESH, ls="--",
               lw=1.2, label=f"Eşik {BEST_THRESHOLD:.3f}")

    # İlk alarm noktasını işaretle
    breach = bdat[bdat["anomaly_score"] > BEST_THRESHOLD]
    if not breach.empty:
        first_t = breach["t"].iloc[0]
        ax.axvline(first_t, color="#FF9F43", ls=":", lw=1.4,
                   label=f"İlk alarm: t={first_t}")

    # Başlık ve doğru/yanlış etiketi
    status = "✓" if lbl == pred else "✗"
    title_color = "#3DD68C" if lbl == pred else "#FF5370"
    ax.set_title(
        f"{bid}  [{'ANORMAL' if lbl==1 else 'NORMAL'}]  {status}",
        fontsize=9, color=title_color
    )
    ax.set_xlabel("t (ölçüm sırası)", fontsize=7)
    ax.set_ylabel("Skor", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.legend(fontsize=6)

for j in range(n_batches, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("anomali_skor_zaman_serileri.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("anomali_skor_zaman_serileri.png")
print("✓ Anomali skor zaman serileri indirildi.")


### HÜCRE 12: Sensör Katkı Analizi (Manuel SHAP Proxy)

In [ ]:
# ── HÜCRE 12: Sensör Katkı Analizi (Manuel SHAP Proxy) ──────────────────────
# SHAP yerine: sensörleri sırayla sıfırlayıp skor değişimine bak

print("Sensör katkı analizi hesaplanıyor (≈1-2 dk)...")

baseline_scores = model.score_samples(X_test_scaled)

sensor_importance = {}
for s_idx, s_name in enumerate(SENSOR_LABELS):
    X_perturbed = X_test_scaled.copy()
    # Bu sensörün tüm penceredeki değerlerini sıfırla (ortalamaya çek)
    for w in range(WINDOW_SIZE):
        feat_idx = w * len(SENSOR_COLS) + s_idx
        X_perturbed[:, feat_idx] = 0.0   # scale edilmiş ortalama = 0
    perturbed_scores = model.score_samples(X_perturbed)
    delta = np.abs(baseline_scores - perturbed_scores).mean()
    sensor_importance[s_name] = delta

imp_df = pd.DataFrame(
    list(sensor_importance.items()),
    columns=["Sensör", "Önem"]
).sort_values("Önem", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Sensör Katkı Analizi", fontsize=13, color="#FFFFFF")

# Genel önem
ax = axes[0]
colors_imp = plt.cm.RdYlGn(
    np.linspace(0.2, 0.9, len(imp_df))
)
bars = ax.barh(imp_df["Sensör"], imp_df["Önem"],
               color=colors_imp, edgecolor="none", alpha=0.85)
ax.set_xlabel("Ortalama Skor Değişimi (Sensör Katkısı)")
ax.set_title("Genel Sensör Önemi")
for bar, val in zip(bars, imp_df["Önem"]):
    ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=8, color="#C8CDD8")

# Anormal partilerde sensör katkısı
ax2 = axes[1]
anom_batches_list = batch_labels[batch_labels["label"]==1]["batch_id"].tolist()
anom_mask = meta_test["batch_id"].isin(anom_batches_list).values

sensor_imp_anom = {}
for s_idx, s_name in enumerate(SENSOR_LABELS):
    X_perturbed = X_test_scaled.copy()
    for w in range(WINDOW_SIZE):
        feat_idx = w * len(SENSOR_COLS) + s_idx
        X_perturbed[:, feat_idx] = 0.0
    perturbed_scores = model.score_samples(X_perturbed)
    delta = np.abs(
        baseline_scores[anom_mask] - perturbed_scores[anom_mask]
    ).mean()
    sensor_imp_anom[s_name] = delta

imp_df2 = pd.DataFrame(
    list(sensor_imp_anom.items()),
    columns=["Sensör", "Önem"]
).sort_values("Önem", ascending=True)

colors_imp2 = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(imp_df2)))
bars2 = ax2.barh(imp_df2["Sensör"], imp_df2["Önem"],
                 color=colors_imp2, edgecolor="none", alpha=0.85)
ax2.set_xlabel("Ortalama Skor Değişimi")
ax2.set_title("Anormal Partilerde Sensör Önemi")
for bar, val in zip(bars2, imp_df2["Önem"]):
    ax2.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", va="center", fontsize=8, color="#C8CDD8")

plt.tight_layout()
plt.savefig("sensor_katkisi.png", dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
files.download("sensor_katkisi.png")
print("✓ Sensör katkı analizi indirildi.")


### HÜCRE 13: Özet Tablo & Tüm Sonuçları CSV İndir

In [ ]:
# ── HÜCRE 13: Özet Tablo & Tüm Sonuçları CSV İndir ─────────────────────────

# Parti bazı özet
summary_df = batch_results.copy()
summary_df["Durum"]       = summary_df["true_label"].map({0: "Normal", 1: "Anormal"})
summary_df["Tahmin"]      = summary_df["pred_label"].map({0: "Normal", 1: "Anormal"})
summary_df["Doğru_mu"]    = summary_df["true_label"] == summary_df["pred_label"]
summary_df["İlk_Alarm_t"] = summary_df["batch_id"].apply(
    lambda bid: (
        meta_test[
            (meta_test["batch_id"] == bid) &
            (meta_test["anomaly_score"] > BEST_THRESHOLD)
        ]["t"].min()
        if not meta_test[
            (meta_test["batch_id"] == bid) &
            (meta_test["anomaly_score"] > BEST_THRESHOLD)
        ].empty
        else None
    )
)
summary_df = summary_df[[
    "batch_id", "Durum", "Tahmin", "Doğru_mu",
    "max_score", "mean_score", "İlk_Alarm_t"
]].rename(columns={
    "batch_id":  "Parti",
    "max_score": "Max_Skor",
    "mean_score":"Ort_Skor",
})

print("\n═" * 28)
print("  PARTİ BAZINDA SONUÇ ÖZETİ")
print("═" * 28)
print(summary_df.to_string(index=False))

# CSV olarak kaydet ve indir
meta_test.to_csv("anomali_skorlari_tam.csv", index=False, encoding="utf-8-sig")
summary_df.to_csv("parti_sonuc_ozeti.csv",   index=False, encoding="utf-8-sig")

files.download("anomali_skorlari_tam.csv")
files.download("parti_sonuc_ozeti.csv")
print("\n✓ Tüm CSV dosyaları indirildi.")

# Model parametrelerini kaydet
model_params = {
    "window_size":    WINDOW_SIZE,
    "n_estimators":   model.n_estimators,
    "contamination":  model.contamination,
    "best_threshold": BEST_THRESHOLD,
    "roc_auc":        round(auc_val, 4),
    "avg_precision":  round(ap, 4),
    "n_train_samples":len(X_train),
    "n_sensors":      len(SENSOR_COLS),
    "sensor_names":   SENSOR_LABELS,
}
import json
with open("model_parametreleri.json", "w", encoding="utf-8") as f:
    json.dump(model_params, f, ensure_ascii=False, indent=2)
files.download("model_parametreleri.json")
print("✓ Model parametreleri indirildi.")

print("\n" + "═" * 55)
print("  TÜM ADIMLAR TAMAMLANDI ✓")
print("═" * 55)
print(f"  İndirilen dosyalar:")
print(f"    📊 eda_genel_bakis.png")
print(f"    📈 eda_zaman_serileri.png")
print(f"    📉 eda_kde_dagilimlar.png")
print(f"    🎯 performans_metrikleri.png")
print(f"    ⏱  anomali_skor_zaman_serileri.png")
print(f"    🔍 sensor_katkisi.png")
print(f"    💾 anomali_skorlari_tam.csv")
print(f"    📋 parti_sonuc_ozeti.csv")
print(f"    ⚙️  model_parametreleri.json")
